[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc5_abtest/exercices/seance1_exercices.ipynb)

# Séance 5.1 — A/B testing — causalité et expériences randomisées

**Exercices** · durée : 6h (2h de cours, 2h d'étude de cas, 2h de correction)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer une question prédictive d'une question causale
- raisonner avec les résultats potentiels, le contrefactuel, l'ATE et l'ATT
- décomposer une comparaison de moyennes en effet causal et biais de sélection
- expliquer ce que la randomisation change, et vérifier qu'elle a fonctionné
- mesurer l'effet d'une campagne, son incertitude, et en tirer une décision chiffrée

## Causalité et A/B testing avec les données Hillstrom

**Durée :** 2 h (1 h 30 de travail + 30 min de correction collective).
**Livrable final :** une recommandation écrite à la direction marketing (Question 13).

**Mode d'emploi :**
- Les questions marquées ★ constituent le chemin obligatoire.
- Les questions ★★ sont pour celles et ceux qui avancent vite.
- Des indices sont cachés dans des blocs repliables — essayez d'abord sans les ouvrir.

> Les données proviennent du *MineThatData E-Mail Analytics Challenge* (Kevin Hillstrom, 2008). Il s'agit d'une **vraie expérience** menée sur 64 000 clients d'un site de e-commerce.

## Le brief

Vous êtes l'équipe data science d'une entreprise de vente en ligne. Il y a deux semaines, l'entreprise a mené une expérience sur 64 000 clients, répartis **au hasard** en trois groupes :

- pas d'email (groupe de contrôle) ;
- un email présentant des produits pour hommes (*Mens E-Mail*) ;
- un email présentant des produits pour femmes (*Womens E-Mail*).

Ce matin, la directrice marketing vous transfère la slide d'un stagiaire, très enthousiaste :

> 💬 *« Résultat spectaculaire : les clients qui ont reçu l'email hommes **et visité le site** dépensent en moyenne plus de **7 euros**, contre environ **0,65** pour le groupe sans email. Nos emails multiplient les dépenses par 10 ! Il faut généraliser immédiatement. »*

Elle vous demande trois choses :

1. **Auditer** l'analyse du stagiaire : ce chiffre est-il un effet causal ?
2. Mesurer l'effet **réel** des deux campagnes sur les visites, les achats et les dépenses.
3. Recommander une stratégie, chiffres à l'appui.

Gardez la slide du stagiaire en tête : vous y reviendrez à la fin.

## Partie 1 — Charger et comprendre les données ★ *(~20 min)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/data/"

In [ ]:
data = pd.read_csv(BASE + "hillstrom.csv")

print("Données chargées :", data.shape[0], "lignes,", data.shape[1], "colonnes")

### Dictionnaire des variables

Variables mesurées **avant** l'expérience (pré-traitement) :

- `recency` : nombre de mois depuis le dernier achat ;
- `history` : montant dépensé pendant l'année précédente ($) ;
- `history_segment` : catégorie de dépense passée ;
- `mens` / `womens` : le client avait acheté des produits hommes / femmes par le passé ;
- `zip_code` : type de zone de résidence ;
- `newbie` : nouveau client (inscrit dans les 12 derniers mois) ;
- `channel` : canal d'achat habituel.

Attribué **pendant** l'expérience :

- `segment` : `No E-Mail`, `Mens E-Mail` ou `Womens E-Mail`.

Mesuré **après**, pendant les deux semaines suivantes :

- `visit` : le client a visité le site (0/1) ;
- `conversion` : le client a acheté (0/1) ;
- `spend` : montant dépensé ($).

⚠️ `mens` et `womens` décrivent les **achats passés**, pas le genre du client.

### Question 1 ★ — Première inspection

Affichez :

1. les cinq premières lignes ;
2. le nombre de valeurs manquantes par colonne ;
3. le nombre de clients dans chaque groupe de `segment`.

<details><summary>💡 Indice</summary>

`head()`, `isna().sum()`, `value_counts()` — vus dans le cours pandas (séance 2).
</details>

In [ ]:
#Vos réponses :

### Question 2 ★ — Traduire le problème dans le langage du cours

Dans le cours, nous avons noté $T_i$ le traitement et $Y_i$ le résultat observé. Répondez en une phrase par question :

1. Dans cette expérience, qu'est-ce que $T_i$ ? (Attention : il y a ici *deux* traitements possibles — on comparera chacun au contrôle.)
2. Si on s'intéresse aux dépenses, qu'est-ce que $Y_i$ ?
3. Le client n° 42 a reçu le *Mens E-Mail* et a dépensé $0$. Que représenterait son $Y_{0,42}$ ? Peut-on l'observer ?
4. Comment s'appelle, dans le cours, ce résultat qu'on ne peut jamais observer ?

**Vos réponses :**

1.
2.
3.
4.

### Question 3 ★ — Prédiction ou causalité ?

Pour chaque question, indiquez s'il s'agit d'une question **prédictive** (séance Machine Learning) ou **causale** (ce cours). Justifiez en une phrase.

1. Quels clients ont le plus de chances d'acheter dans les deux prochaines semaines ?
2. Envoyer un email augmente-t-il la probabilité d'achat ?
3. Combien un client va-t-il probablement dépenser ?
4. Quelle serait la dépense des clients si l'entreprise leur envoyait l'email hommes plutôt qu'aucun email ?

**Vos réponses :**

1.
2.
3.
4.

## Partie 2 — Le piège : un monde sans randomisation ★ *(~20 min)*

Avant d'analyser la vraie expérience, faisons un détour instructif.

**Imaginons un monde parallèle** où l'entreprise n'aurait *pas* randomisé. Comme beaucoup d'entreprises, elle aurait envoyé l'email en priorité à ses **meilleurs clients** — ceux qui ont beaucoup dépensé l'année précédente. C'est exactement l'histoire des tablettes dans les écoles riches, version marketing.

La cellule ci-dessous **simule** ce monde parallèle à partir de nos données : on garde les clients *Mens E-Mail* dont la dépense passée (`history`) est au-dessus de la médiane, et les clients *No E-Mail* en dessous. Exécutez-la simplement.

In [ ]:
# Simulation d'une base "observationnelle" (monde parallèle, PAS la vraie expérience)
mediane = data["history"].median()

monde_parallele = pd.concat([
    data[(data["segment"] == "Mens E-Mail") & (data["history"] > mediane)],
    data[(data["segment"] == "No E-Mail") & (data["history"] <= mediane)]
])

monde_parallele["recu_email"] = (monde_parallele["segment"] == "Mens E-Mail").astype(int)
print("Base simulée :", monde_parallele.shape[0], "clients")

### Question 4 ★ — L'estimation naïve

Dans ce monde parallèle, calculez la dépense moyenne (`spend`) des clients ayant reçu l'email et celle des clients sans email, puis leur différence.

**Notez précieusement ce chiffre** : c'est ce qu'un analyste pressé appellerait « l'effet de l'email ». Nous le confronterons à la vérité en Partie 4.

<details><summary>💡 Indice</summary>

`monde_parallele.groupby("recu_email")["spend"].mean()` puis une soustraction.
</details>

In [ ]:
# À vous de jouer

### Question 5 ★ — Diagnostiquer le biais

1. Toujours dans `monde_parallele`, calculez la moyenne de `history` (dépense passée) pour chacun des deux groupes. Que constatez-vous ?
2. Le cours décompose la comparaison naïve ainsi :

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
\underset{\mathrm{ATT}}
{\Bigl(E[Y_1-Y_0 \mid T=1]\Bigr)}
+
\underset{\text{Biais de sélection}}
{\Bigl(E[Y_0 \mid T=1]-E[Y_0 \mid T=0]\Bigr)}.
$$


En une ou deux phrases : dans ce monde parallèle, pourquoi $E[Y_0 \mid T=1] \neq E[Y_0 \mid T=0]$ ? Le biais est-il positif ou négatif ici ?



3. Reformulez avec l'exemple du cours : qui joue le rôle des « écoles riches » ? qui joue le rôle des « tablettes » ?

In [ ]:
# Question 5.1 — comparez `history` entre les deux groupes

**Vos réponses (5.2 et 5.3) :**

...

## Partie 3 — Retour au monde réel : la randomisation a-t-elle fonctionné ? ★ *(~10 min)*

Bonne nouvelle : dans la **vraie** expérience, les emails ont été attribués au hasard. La randomisation garantit *en espérance* que $E[Y_0 \mid T=1] = E[Y_0 \mid T=0]$ : les groupes sont comparables avant traitement, le biais de sélection disparaît.

Mais un bon data scientist **vérifie** au lieu de croire sur parole. Si la randomisation a fonctionné, les variables *pré-traitement* doivent être quasiment identiques d'un groupe à l'autre.

### Question 6 ★ — Le test d'équilibre

Complétez le tableau ci-dessous, puis répondez :

1. Les groupes de la vraie expérience vous semblent-ils comparables ?
2. Des valeurs *parfaitement* identiques sont-elles nécessaires ? Pourquoi ?
3. Refaites le même tableau sur `monde_parallele` (en groupant par `recu_email`). Qu'est-ce qui saute aux yeux ? À quoi ce test aurait-il servi si on vous avait donné cette base en vous disant qu'elle était randomisée ?

In [ ]:
# Complétez les noms des colonnes
equilibre = data.groupby("segment").agg(
    nombre_clients=("segment", "size"),
    dépese_moyenne=("...", "mean"),
    depense_passee_moyenne=("...", "mean"),
    proportion_nouveaux=("...", "mean")
).round(3)

equilibre

In [ ]:
# Question 6.3 — même tableau sur monde_parallele

**Votre interprétation :**

...

## Partie 4 — Mesurer l'effet réel des campagnes ★ *(~20 min)*

Puisque les groupes sont comparables, le cours nous dit que la simple différence de moyennes est un effet causal :

$$E[Y \mid T=1]-E[Y \mid T=0]=E[Y_1-Y_0]=ATE$$

### Question 7 ★ — Résultats moyens par groupe

Construisez le tableau des résultats : pour chaque groupe, le nombre de clients, le taux de visite, le taux de conversion et la dépense moyenne.

*Rappel :* pour une variable valant 0 ou 1, la moyenne est la proportion de 1.

In [ ]:
# Complétez les noms des colonnes et les opérations
resultats = data.groupby("segment").agg(
    nombre_clients=("segment", "size"),
    taux_visite=("...", "..."),
    taux_conversion=("...", "..."),
    depense_moyenne=("...", "...")
)

resultats.round(4)

### Question 8 ★ — Visualiser

Créez trois graphiques en barres (un par indicateur) à partir du tableau `resultats`.

<details><summary>💡 Indice</summary>

Le plus simple : partir du tableau agrégé, par exemple `resultats["taux_visite"].plot(kind="bar")`.
Si vous préférez `sns.barplot(data=data, ...)`, ajoutez `errorbar=None` : sinon seaborn recalcule des intervalles sur 64 000 lignes et c'est lent.
</details>

In [ ]:
# À vous de jouer

### Question 9 ★ — Effet absolu, effet relatif… et taille du biais

Pour chaque campagne, calculez son effet sur la conversion et sur la dépense, par rapport au groupe `No E-Mail` :

$$\text{effet absolu}=\text{résultat du traitement}-\text{résultat du contrôle}
\qquad
\text{effet relatif}=\frac{\text{effet absolu}}{\text{résultat du contrôle}}$$

Puis :

1. Expliquez pourquoi un même résultat peut se décrire par un effet absolu modeste **et** un effet relatif spectaculaire. (Exemple générique : passer d'un taux de 1 % à 2 %, c'est +1 point de pourcentage… et +100 %.) Laquelle des deux formulations choisiriez-vous pour une slide honnête ?
2. **Le moment de vérité :** comparez l'effet réel du *Mens E-Mail* sur la dépense à votre estimation naïve de la Question 4. De combien le monde parallèle surestimait-il l'effet ? Ce chiffre, c'est le biais de sélection — en dollars.

In [ ]:
# Exemple de structure — complétez
controle_conversion = data.loc[data["segment"] == "No E-Mail", "conversion"].mean()
controle_depense = data.loc[data["segment"] == "No E-Mail", "spend"].mean()

# À compléter pour chaque campagne

**Votre interprétation :**

- Email hommes : ...
- Email femmes : ...
- Taille du biais (Q9.2) : ...

## Partie 5 — Le résultat peut-il être dû au hasard ? ★ *(~20 min)*

Même avec une randomisation parfaite, deux groupes ne sont jamais *exactement* identiques : le hasard crée de petites différences. Comment savoir si l'effet mesuré dépasse ce que le hasard seul produirait ?

Les fonctions ci-dessous (vues en séance 3) calculent l'effet estimé, un **intervalle de confiance à 95 %** et une **p-value**. Vous n'avez pas à les programmer : exécutez la cellule, puis utilisez-les.

Convention pour cet exercice : une `p_value < 0.05` apporte des éléments contre l'hypothèse « aucun effet ». Elle ne dit **rien** de la taille ni de l'intérêt économique de l'effet.

In [ ]:
from scipy.stats import norm, ttest_ind

def comparer_taux(data, traitement, controle, variable, colonne_groupe="segment"):
    groupe_t = data.loc[data[colonne_groupe] == traitement, variable]
    groupe_c = data.loc[data[colonne_groupe] == controle, variable]
    p_t, p_c = groupe_t.mean(), groupe_c.mean()
    effet = p_t - p_c
    erreur_standard = np.sqrt(
        p_t * (1 - p_t) / len(groupe_t)
        + p_c * (1 - p_c) / len(groupe_c)
    )
    z = effet / erreur_standard
    p_value = 2 * norm.sf(abs(z))
    return pd.Series({
        "effet": effet,
        "borne_basse_95": effet - 1.96 * erreur_standard,
        "borne_haute_95": effet + 1.96 * erreur_standard,
        "p_value": p_value
    })

def comparer_moyennes(data, traitement, controle, variable, colonne_groupe="segment"):
    groupe_t = data.loc[data[colonne_groupe] == traitement, variable]
    groupe_c = data.loc[data[colonne_groupe] == controle, variable]
    effet = groupe_t.mean() - groupe_c.mean()
    erreur_standard = np.sqrt(
        groupe_t.var(ddof=1) / len(groupe_t)
        + groupe_c.var(ddof=1) / len(groupe_c)
    )
    p_value = ttest_ind(groupe_t, groupe_c, equal_var=False).pvalue
    return pd.Series({
        "effet": effet,
        "borne_basse_95": effet - 1.96 * erreur_standard,
        "borne_haute_95": effet + 1.96 * erreur_standard,
        "p_value": p_value
    })

### Question 10 ★ — Tester les effets

Exécutez les quatre comparaisons :

1. email hommes vs aucun email, sur la conversion (`comparer_taux`) ;
2. email femmes vs aucun email, sur la conversion ;
3. email hommes vs aucun email, sur la dépense (`comparer_moyennes`) ;
4. email femmes vs aucun email, sur la dépense.

Pour chacune : l'intervalle de confiance contient-il zéro ? Que concluez-vous ?

In [ ]:
# Exemple
comparer_taux(data, "Mens E-Mail", "No E-Mail", "conversion")

# Ajoutez les trois autres comparaisons

**Votre interprétation :**

...

### Question 11 ★ — Quelle campagne est la meilleure ?

Attention au raccourci : « A bat le contrôle, B bat le contrôle, et l'effet de A est plus grand, donc A bat B ». Pour affirmer que A bat B, il faut comparer **A à B directement**.

Comparez `Mens E-Mail` à `Womens E-Mail` sur le taux de conversion et sur la dépense moyenne. La différence entre les deux campagnes est-elle statistiquement détectable dans les deux cas ?

In [ ]:
# À vous de jouer, en réutilisant les fonctions précédentes

### Question 12 ★★ — L'expérience du petit échantillon

Votre concurrent, plus petit, n'a que 2 000 clients pour faire le même test. Simulez sa situation :

In [ ]:
petit_echantillon = data.sample(2000, random_state=1)

# Relancez la comparaison Mens E-Mail vs No E-Mail sur la conversion,
# mais sur `petit_echantillon` au lieu de `data`

1. Comparez l'effet estimé, la largeur de l'intervalle et la p-value avec vos résultats de la Question 10.
2. L'effet de l'email a-t-il « disparu » chez le concurrent ? Que peut-il conclure — et surtout, que ne peut-il **pas** conclure ?
3. Essayez plusieurs valeurs de `random_state`. Que constatez-vous sur la stabilité de l'effet estimé ?

**Vos réponses :**

...

## Partie 6 — De la statistique à la décision ★ *(~20 min)*

### Question 13a ★ — La campagne est-elle rentable ?

La directrice marketing vous donne les paramètres suivants :

- l'entreprise peut contacter 100 000 clients ;
- la marge représente 40 % du chiffre d'affaires ;
- chaque email envoyé coûte 0,05 $.

Pour chaque campagne, calculez :

1. l'effet sur la dépense par client (déjà obtenu en Question 10) ;
2. la marge supplémentaire par client ;
3. le bénéfice net supplémentaire par client ;
4. le bénéfice net attendu pour 100 000 clients.

$$\text{bénéfice net par client}=\text{effet sur la dépense}\times\text{taux de marge}-\text{coût de l'email}$$

★★ *Pour aller plus loin :* refaites le calcul 4 avec les bornes basse et haute de l'intervalle de confiance de l'effet. Quelle fourchette de bénéfice obtenez-vous ? Est-elle confortable pour décider ?

In [ ]:
nombre_clients_cibles = 100_000
taux_marge = 0.40
cout_email = 0.05

# À vous de jouer

### Question 13b ★ — Note à la direction (livrable final)

Rédigez votre recommandation en **cinq phrases maximum**. Elle sera jugée sur cette checklist :

- [ ] une stratégie claire est recommandée ;
- [ ] l'effet estimé sur la conversion et la dépense est cité (en choisissant honnêtement entre absolu et relatif) ;
- [ ] l'incertitude statistique est mentionnée (intervalle, pas seulement « significatif ») ;
- [ ] l'ordre de grandeur du bénéfice attendu apparaît ;
- [ ] au moins une limite de l'analyse est signalée ;
- [ ] zéro jargon inutile : la directrice marketing n'a jamais entendu parler de p-value.

**Votre recommandation :**

...

## Partie 7 — Questions de recul ★ *(~10 min, sans coder)*

1. Pourquoi peut-on interpréter ici la différence entre les groupes comme un effet causal, alors qu'on ne le pouvait pas dans le monde parallèle de la Partie 2 ?
2. Une p-value inférieure à 0,05 suffit-elle pour décider d'envoyer l'email ? Qu'est-ce qui manque ?
3. **Le stagiaire revient.** Nouvelle idée : « Comparons la dépense des clients qui ont visité le site à celle des clients qui n'ont pas visité : on verra l'effet causal de la visite ! » `visit` est-elle une variable pré-traitement ou post-traitement ? Pourquoi cette comparaison retombe-t-elle exactement dans le piège de la Partie 2 ? (C'était d'ailleurs l'erreur de sa slide initiale : il comparait les *visiteurs* traités au contrôle entier.)
4. L'expérience date de 2008, sur un site américain. Peut-on garantir les mêmes résultats en France en 2026 ? Comment s'appelle ce problème ?
5. Avant de lancer une nouvelle expérience, que doit fixer l'entreprise : le KPI principal, les segments analysés et la durée du test — ou peut-elle attendre les résultats pour choisir ? Pourquoi ?

**Vos réponses :**

1.
2.
3.
4.
5.

## Bonus ★★ — Segmentation exploratoire

Les campagnes semblent-elles avoir le même effet selon le canal d'achat habituel (`channel`) ? Calculez la dépense moyenne par `channel` et par `segment`, et repérez le segment où l'écart semble le plus grand.

⚠️ Cette analyse est **exploratoire** : plus on découpe les données en segments, plus on risque de prendre une fluctuation due au hasard pour un véritable effet (le problème des comparaisons multiples, cousin de votre Question 12). Toute hypothèse intéressante devrait être confirmée par une **nouvelle** expérience dédiée. C'est exactement le sens de votre réponse à la question de recul n°5.

In [ ]:
# Bonus

---

**Source des données :** Kevin Hillstrom, *The MineThatData E-Mail Analytics And Data Mining Challenge*, 2008.
https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html